# 01_data_collection_mop.ipynb
## Project Title: Traffic Accident Risk Prediction (TARP)

**Unit:** SIT782  
**Prepared by:** Subathira Thinakaran  

**Project Team:**  
- Suba (225094537)  
- Burhan (224802775)  
- Khalid (224696667)  

**Task:** Data Collection – Import and Explore MOP Pedestrian and Bicycle Datasets (Week 1 of 8)

## Objective
This notebook implements a data collection pipeline using the City of Melbourne Open Data API.  
It retrieves pedestrian and bicycle datasets, validates their structure, and stores them locally for further preprocessing and analysis.

## Datasets Used
- Pedestrian Counting System (Hourly Data)
- Annual Bike Counts (Super Tuesday)

## Output
- pedestrian_raw.csv  
- bicycle_raw.csv  

## API Key Setup

This notebook uses the City of Melbourne Open Data API.

To run this notebook:
1. Open the "Secrets" tab in Google Colab (🔑 icon)
2. Add a new secret:
   - Name: MOP_API_KEY
   - Value: Your API key
3. Run the notebook

Note: API keys are not stored in this notebook for security reasons.

## API Configuration

This section defines:
- API base URL  
- Dataset identifiers  
- Secure API key access  
- Local storage path  

In [24]:
import requests
import pandas as pd
from pathlib import Path
from google.colab import userdata

# -----------------------------
# 1. API configuration
# -----------------------------
BASE_URL = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets"

PEDESTRIAN_DATASET = "pedestrian-counting-system-monthly-counts-per-hour"
BICYCLE_DATASET = "annual-bike-counts-super-tuesday"

API_KEY = userdata.get("MOP_API_KEY")
if not API_KEY:
    raise ValueError("MOP_API_KEY not found in Colab secrets.")

RAW_DATA_DIR = Path("data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("API base URL:", BASE_URL)
print("Pedestrian dataset:", PEDESTRIAN_DATASET)
print("Bicycle dataset:", BICYCLE_DATASET)
print("API key loaded:", API_KEY is not None)

API base URL: https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets
Pedestrian dataset: pedestrian-counting-system-monthly-counts-per-hour
Bicycle dataset: annual-bike-counts-super-tuesday
API key loaded: True


## Helper Functions

This section defines reusable functions to interact with the MOP API.  
`fetch_records()` retrieves a single batch of records, while `fetch_all_records()` handles pagination to collect the full dataset efficiently.

In [33]:
# -----------------------------
# 2. Helper functions
# -----------------------------
import time

def fetch_records(dataset_id, limit=100, offset=0, where=None, select=None, order_by=None):
    """
    Fetch one page of records from the City of Melbourne Open Data API.
    """
    headers = {
        "Authorization": f"Apikey {API_KEY}"
    }

    url = f"{BASE_URL}/{dataset_id}/records"
    params = {
        "limit": limit,
        "offset": offset
    }

    if where:
        params["where"] = where
    if select:
        params["select"] = select
    if order_by:
        params["order_by"] = order_by

    try:
        response = requests.get(url, params=params, headers=headers, timeout=60)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from {dataset_id}: {e}")
        return {}


def fetch_all_records(dataset_id, batch_size=100, where=None, select=None, order_by=None, max_records=None):
    """
    Fetch all records from a dataset using pagination.
    """
    all_rows = []
    offset = 0

    while True:
        payload = fetch_records(
            dataset_id=dataset_id,
            limit=batch_size,
            offset=offset,
            where=where,
            select=select,
            order_by=order_by
        )

        rows = payload.get("results", [])
        if not rows:
            break

        all_rows.extend(rows)
        offset += batch_size

        if len(all_rows) % 1000 == 0 or (max_records is not None and len(all_rows) >= max_records):
            print(f"Fetched {len(all_rows)} records from {dataset_id}")

        if max_records is not None and len(all_rows) >= max_records:
            all_rows = all_rows[:max_records]
            break

        time.sleep(0.1)

    return pd.DataFrame(all_rows)

## API Connectivity Test

This step verifies that the API is accessible and returning valid data.  
A small sample of records is retrieved to confirm successful communication and response structure.

In [34]:
# -----------------------------
# 3. Test API connectivity
# -----------------------------
test_json = fetch_records(
    dataset_id=PEDESTRIAN_DATASET,
    limit=5
)

print("Keys in response:", test_json.keys())
print("Number of preview rows:", len(test_json.get("results", [])))

# Preview sample data
pd.DataFrame(test_json["results"]).head()

Keys in response: dict_keys(['total_count', 'results'])
Number of preview rows: 5


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location
0,431320241029,43,2024-10-29,13,152,137,289,UM2_T,"{'lon': 144.96411782, 'lat': -37.79844526}"
1,451220250220,45,2025-02-20,12,483,818,1301,Swa148_T,"{'lon': 144.96609379, 'lat': -37.81414075}"
2,452320260210,45,2026-02-10,23,117,169,286,Swa148_T,"{'lon': 144.96609379, 'lat': -37.81414075}"
3,1671720260111,167,2026-01-11,17,28,22,50,Lat526_T,"{'lon': 144.95155955, 'lat': -37.81304149}"
4,1622020251008,162,2025-10-08,20,422,517,939,Eli489_T,"{'lon': 144.95987909, 'lat': -37.80740397}"


## Pedestrian Data Preview

Convert the API response into a pandas DataFrame and inspect the structure and sample records of the pedestrian dataset.

In [35]:
# -----------------------------
# 4. Preview pedestrian data
# -----------------------------
pedestrian_preview = pd.DataFrame(test_json["results"])

print("Preview shape:", pedestrian_preview.shape)
print("Columns:", list(pedestrian_preview.columns))

pedestrian_preview.head()

Preview shape: (5, 9)
Columns: ['id', 'location_id', 'sensing_date', 'hourday', 'direction_1', 'direction_2', 'pedestriancount', 'sensor_name', 'location']


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location
0,431320241029,43,2024-10-29,13,152,137,289,UM2_T,"{'lon': 144.96411782, 'lat': -37.79844526}"
1,451220250220,45,2025-02-20,12,483,818,1301,Swa148_T,"{'lon': 144.96609379, 'lat': -37.81414075}"
2,452320260210,45,2026-02-10,23,117,169,286,Swa148_T,"{'lon': 144.96609379, 'lat': -37.81414075}"
3,1671720260111,167,2026-01-11,17,28,22,50,Lat526_T,"{'lon': 144.95155955, 'lat': -37.81304149}"
4,1622020251008,162,2025-10-08,20,422,517,939,Eli489_T,"{'lon': 144.95987909, 'lat': -37.80740397}"


### Observation

The dataset contains temporal (date and hour), directional counts, and location-based information.  
The `pedestriancount` field represents total pedestrian activity, while the `location` field contains geographic coordinates that can be used for spatial analysis.

## Full Pedestrian Dataset Import

This step retrieves the pedestrian dataset using pagination to handle large volumes of data.  
A subset of records is loaded during development for efficiency.

In [37]:
# -----------------------------
# 5. Import pedestrian dataset
# -----------------------------
# During development, a subset of records is loaded for efficiency.
# The full dataset can be retrieved later by removing max_records.

pedestrian_df = fetch_all_records(
    dataset_id=PEDESTRIAN_DATASET,
    batch_size=100,
    max_records=5000
)

print("Pedestrian dataset shape:", pedestrian_df.shape)
print("Number of unique sensors:", pedestrian_df["sensor_name"].nunique())

pedestrian_df.head()

Fetched 1000 records from pedestrian-counting-system-monthly-counts-per-hour
Fetched 2000 records from pedestrian-counting-system-monthly-counts-per-hour
Fetched 3000 records from pedestrian-counting-system-monthly-counts-per-hour
Fetched 4000 records from pedestrian-counting-system-monthly-counts-per-hour
Fetched 5000 records from pedestrian-counting-system-monthly-counts-per-hour
Pedestrian dataset shape: (5000, 9)
Number of unique sensors: 98


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location
0,431320241029,43,2024-10-29,13,152,137,289,UM2_T,"{'lon': 144.96411782, 'lat': -37.79844526}"
1,451220250220,45,2025-02-20,12,483,818,1301,Swa148_T,"{'lon': 144.96609379, 'lat': -37.81414075}"
2,452320260210,45,2026-02-10,23,117,169,286,Swa148_T,"{'lon': 144.96609379, 'lat': -37.81414075}"
3,1671720260111,167,2026-01-11,17,28,22,50,Lat526_T,"{'lon': 144.95155955, 'lat': -37.81304149}"
4,1622020251008,162,2025-10-08,20,422,517,939,Eli489_T,"{'lon': 144.95987909, 'lat': -37.80740397}"


### Observation

A total of 5000 pedestrian records were successfully imported from the MOP API.  
The dataset contains 98 unique sensors and includes temporal, directional, and geographic attributes, which will support later cleaning, exploratory analysis, and feature engineering.

## Bicycle Dataset Preview

Retrieve a sample of the bicycle dataset using the MOP API and convert it into a pandas DataFrame.  
This step helps in understanding the structure, features, and complexity of the dataset.

In [40]:
# -----------------------------
# 6. Preview bicycle dataset
# -----------------------------
bicycle_preview = pd.DataFrame(fetch_records(BICYCLE_DATASET, limit=5)["results"])

print("Bicycle preview shape:", bicycle_preview.shape)
print("Total columns:", len(bicycle_preview.columns))
print("Columns:", list(bicycle_preview.columns))

key_columns = ["site_id", "latitude", "longitude", "total", "year"]
print("\nKey columns:", key_columns)
print("\nData types of key columns:")
print(bicycle_preview[key_columns].dtypes)

bicycle_preview.head()

Bicycle preview shape: (5, 82)
Total columns: 82
Columns: ['state', 'electorate', 'site_id', 'latitude', 'longitude', 'legs', 'description', 'layout_1', 'layout_1_enter', 'layout_2', 'layout_2_enter', 'layout_3', 'layout_3_enter', 'layout_4', 'layout_4_enter', 'layout_5', 'layout_5_enter', 'layout_6', 'layout_6_enter', 'leg1_2', 'leg1_3', 'leg1_4', 'leg1_5', 'leg1_6', 'leg2_1', 'leg2_3', 'leg2_4', 'leg2_5', 'leg2_6', 'leg3_1', 'leg3_2', 'leg3_4', 'leg3_5', 'leg3_6', 'leg4_1', 'leg4_2', 'leg4_3', 'leg4_5', 'leg4_6', 'leg5_1', 'leg5_2', 'leg5_3', 'leg5_4', 'leg5_6', 'leg6_1', 'leg6_2', 'leg6_3', 'leg6_4', 'leg6_5', 'leg1_enter', 'leg1_exit', 'leg1_total', 'leg2_enter', 'leg2_exit', 'leg2_total', 'leg3_enter', 'leg3_exit', 'leg3_total', 'leg4_enter', 'leg4_exit', 'leg4_total', 'leg5_enter', 'leg5_exit', 'leg5_total', 'leg6_enter', 'leg6_exit', 'leg6_total', 'female', 'male', 'not_known', 'total', 'year', '7_00_am', '7_15_am', '7_30_am', '7_45_am', '8_00_am', '8_15_am', '8_30_am', '8_45_am

,state,electorate,site_id,latitude,longitude,legs,description,layout_1,layout_1_enter,layout_2,...,7_00_am,7_15_am,7_30_am,7_45_am,8_00_am,8_15_am,8_30_am,8_45_am,location,geolocation
0,VIC,Melbourne,4411,-37.796627,144.951172,4,"Gatehouse St [NE], Flemington Rd [SE], Harker ...",45,225,135,...,24,36,48,47,72,84,80,91,POINT (144.951172 -37.796627),"{'lon': 144.951172, 'lat': -37.796627}"
1,VIC,Melbourne,4417,-37.808903,144.966278,4,"La Trobe St [E], Russell St (city) [S], La Tro...",70,250,160,...,29,56,73,76,120,129,146,113,POINT (144.966278 -37.808903),"{'lon': 144.966278, 'lat': -37.808903}"
2,VIC,Melbourne,4418,-37.809248,144.972971,4,"Nicholson St [N], Albert St [E], Nicholson St ...",8,188,98,...,26,43,52,73,79,102,90,97,POINT (144.972971 -37.809248),"{'lon': 144.972971, 'lat': -37.809248}"
3,VIC,Melbourne,4425,-37.811039,144.958847,4,"La Trobe towards Exhibition St [E], Queen St [...",70,250,160,...,35,52,60,93,111,127,105,112,POINT (144.958847 -37.811039),"{'lon': 144.958847, 'lat': -37.811039}"
4,VIC,Melbourne,4436,-37.817448,144.967438,4,"Flinders St [E], Swanston St [S], Flinders St ...",70,250,160,...,104,99,221,237,279,309,316,226,POINT (144.967438 -37.817448),"{'lon': 144.967438, 'lat': -37.817448}"


### Observation

The bicycle dataset contains 82 columns, including spatial, directional, demographic, and time-based features. Key fields such as `site_id`, `latitude`, `longitude`, `total`, and `year` appear useful for later analysis. Compared to the pedestrian dataset, this dataset has a more complex structure and will require careful cleaning and feature selection before integration.

## Bicycle Dataset Columns

Inspect all column names in the bicycle dataset to understand the available features and identify relevant variables for analysis.

In [41]:
# -----------------------------
# 7. Check bicycle columns
# -----------------------------
print(f"Total number of columns: {len(bicycle_preview.columns)}\n")

print("Bicycle dataset columns:")
for col in bicycle_preview.columns:
    print("-", col)

Total number of columns: 82

Bicycle dataset columns:
- state
- electorate
- site_id
- latitude
- longitude
- legs
- description
- layout_1
- layout_1_enter
- layout_2
- layout_2_enter
- layout_3
- layout_3_enter
- layout_4
- layout_4_enter
- layout_5
- layout_5_enter
- layout_6
- layout_6_enter
- leg1_2
- leg1_3
- leg1_4
- leg1_5
- leg1_6
- leg2_1
- leg2_3
- leg2_4
- leg2_5
- leg2_6
- leg3_1
- leg3_2
- leg3_4
- leg3_5
- leg3_6
- leg4_1
- leg4_2
- leg4_3
- leg4_5
- leg4_6
- leg5_1
- leg5_2
- leg5_3
- leg5_4
- leg5_6
- leg6_1
- leg6_2
- leg6_3
- leg6_4
- leg6_5
- leg1_enter
- leg1_exit
- leg1_total
- leg2_enter
- leg2_exit
- leg2_total
- leg3_enter
- leg3_exit
- leg3_total
- leg4_enter
- leg4_exit
- leg4_total
- leg5_enter
- leg5_exit
- leg5_total
- leg6_enter
- leg6_exit
- leg6_total
- female
- male
- not_known
- total
- year
- 7_00_am
- 7_15_am
- 7_30_am
- 7_45_am
- 8_00_am
- 8_15_am
- 8_30_am
- 8_45_am
- location
- geolocation


### Observation

The bicycle dataset contains 82 columns, which can be grouped into:
- Spatial features: latitude, longitude, location  
- Directional movement features: leg1, leg2, etc.  
- Aggregated counts: total, male, female  
- Temporal features: year and time intervals (7:00 AM, etc.)

This indicates that the dataset is highly detailed and will require feature selection and transformation during the data cleaning stage.

## Full Bicycle Dataset Import

This step retrieves the bicycle dataset using the MOP API.  
Since the dataset is relatively small, all available records are loaded using pagination.

In [43]:
# -----------------------------
# 8. Import bicycle dataset
# -----------------------------
# annual-bike-counts-super-tuesday is already a bike-only dataset,
# so no filtering is required.

bicycle_df = fetch_all_records(
    dataset_id=BICYCLE_DATASET,
    batch_size=100,
    max_records=5000
)

print("Bicycle dataset shape:", bicycle_df.shape)
print("Number of unique sites:", bicycle_df["site_id"].nunique())

bicycle_df.head()

Bicycle dataset shape: (352, 82)
Number of unique sites: 46


,state,electorate,site_id,latitude,longitude,legs,description,layout_1,layout_1_enter,layout_2,...,7_00_am,7_15_am,7_30_am,7_45_am,8_00_am,8_15_am,8_30_am,8_45_am,location,geolocation
0,VIC,Melbourne,4411,-37.796627,144.951172,4,"Gatehouse St [NE], Flemington Rd [SE], Harker ...",45,225,135,...,24,36,48,47,72,84,80,91,POINT (144.951172 -37.796627),"{'lon': 144.951172, 'lat': -37.796627}"
1,VIC,Melbourne,4417,-37.808903,144.966278,4,"La Trobe St [E], Russell St (city) [S], La Tro...",70,250,160,...,29,56,73,76,120,129,146,113,POINT (144.966278 -37.808903),"{'lon': 144.966278, 'lat': -37.808903}"
2,VIC,Melbourne,4418,-37.809248,144.972971,4,"Nicholson St [N], Albert St [E], Nicholson St ...",8,188,98,...,26,43,52,73,79,102,90,97,POINT (144.972971 -37.809248),"{'lon': 144.972971, 'lat': -37.809248}"
3,VIC,Melbourne,4425,-37.811039,144.958847,4,"La Trobe towards Exhibition St [E], Queen St [...",70,250,160,...,35,52,60,93,111,127,105,112,POINT (144.958847 -37.811039),"{'lon': 144.958847, 'lat': -37.811039}"
4,VIC,Melbourne,4436,-37.817448,144.967438,4,"Flinders St [E], Swanston St [S], Flinders St ...",70,250,160,...,104,99,221,237,279,309,316,226,POINT (144.967438 -37.817448),"{'lon': 144.967438, 'lat': -37.817448}"


### Observation

A total of 352 records were retrieved, indicating that the full bicycle dataset has been successfully loaded.  
The dataset contains detailed spatial, directional, demographic, and time-based features across multiple sites.  

Compared to the pedestrian dataset, this dataset is smaller in size but more complex in structure, requiring careful feature selection and transformation during the data cleaning stage.

## Basic Validation

This step validates the structure of both datasets by inspecting their shapes and column names.  
It also highlights differences in dataset size and complexity.

In [45]:
# -----------------------------
# 9. Basic validation
# -----------------------------
print("Pedestrian dataset shape:", pedestrian_df.shape)
print("Number of pedestrian columns:", len(pedestrian_df.columns))

print("\nBicycle dataset shape:", bicycle_df.shape)
print("Number of bicycle columns:", len(bicycle_df.columns))

print("\nPedestrian columns:")
for col in pedestrian_df.columns:
    print("-", col)

print("\nBicycle columns:")
for col in bicycle_df.columns:
    print("-", col)

Pedestrian dataset shape: (5000, 9)
Number of pedestrian columns: 9

Bicycle dataset shape: (352, 82)
Number of bicycle columns: 82

Pedestrian columns:
- id
- location_id
- sensing_date
- hourday
- direction_1
- direction_2
- pedestriancount
- sensor_name
- location

Bicycle columns:
- state
- electorate
- site_id
- latitude
- longitude
- legs
- description
- layout_1
- layout_1_enter
- layout_2
- layout_2_enter
- layout_3
- layout_3_enter
- layout_4
- layout_4_enter
- layout_5
- layout_5_enter
- layout_6
- layout_6_enter
- leg1_2
- leg1_3
- leg1_4
- leg1_5
- leg1_6
- leg2_1
- leg2_3
- leg2_4
- leg2_5
- leg2_6
- leg3_1
- leg3_2
- leg3_4
- leg3_5
- leg3_6
- leg4_1
- leg4_2
- leg4_3
- leg4_5
- leg4_6
- leg5_1
- leg5_2
- leg5_3
- leg5_4
- leg5_6
- leg6_1
- leg6_2
- leg6_3
- leg6_4
- leg6_5
- leg1_enter
- leg1_exit
- leg1_total
- leg2_enter
- leg2_exit
- leg2_total
- leg3_enter
- leg3_exit
- leg3_total
- leg4_enter
- leg4_exit
- leg4_total
- leg5_enter
- leg5_exit
- leg5_total
- leg6_ente

### Observation

The pedestrian dataset contains fewer columns (9) with a simpler structure focused on time-based counts and locations.  

In contrast, the bicycle dataset contains significantly more features (82 columns), including directional movement, demographic data, and time-based intervals.  

This highlights a key difference in complexity between the two datasets, indicating that additional preprocessing and feature selection will be required before combining or analysing them.

## Missing Values Analysis

This step checks for missing values in both datasets to identify data quality issues and determine necessary cleaning steps.

In [46]:
# -----------------------------
# 10. Missing values check
# -----------------------------
print("Missing values in pedestrian dataset:")
print(pedestrian_df.isna().sum())

print("\nTop missing values in bicycle dataset:")
missing_bicycle = bicycle_df.isna().sum().sort_values(ascending=False)
print(missing_bicycle.head(10))

# Count fully empty columns
empty_columns = missing_bicycle[missing_bicycle == len(bicycle_df)]
print(f"\nNumber of completely empty columns in bicycle dataset: {len(empty_columns)}")

Missing values in pedestrian dataset:
id                 0
location_id        0
sensing_date       0
hourday            0
direction_1        0
direction_2        0
pedestriancount    0
sensor_name        0
location           0
dtype: int64

Top missing values in bicycle dataset:
layout_6          352
layout_6_enter    352
leg4_6            352
leg6_2            352
leg6_total        352
leg6_enter        352
leg6_exit         352
leg6_3            352
leg6_1            352
leg5_6            352
dtype: int64

Number of completely empty columns in bicycle dataset: 15


### Observation

The pedestrian dataset does not contain any missing values, indicating good data quality and readiness for further analysis.  

In contrast, the bicycle dataset contains several columns with all values missing (e.g., layout_6, leg6_* features). These columns do not provide useful information and will be removed during the data cleaning stage.  

This highlights the need for feature selection and cleaning to reduce dataset complexity and improve model performance.

## Save Raw Datasets

This step saves the collected datasets locally as CSV files for further preprocessing and analysis in subsequent stages.

In [47]:
# -----------------------------
# 11. Save raw datasets
# -----------------------------
pedestrian_file = RAW_DATA_DIR / "pedestrian_raw.csv"
bicycle_file = RAW_DATA_DIR / "bicycle_raw.csv"

pedestrian_df.to_csv(pedestrian_file, index=False)
bicycle_df.to_csv(bicycle_file, index=False)

print("Saved pedestrian data to:", pedestrian_file)
print("Pedestrian records saved:", len(pedestrian_df))

print("\nSaved bicycle data to:", bicycle_file)
print("Bicycle records saved:", len(bicycle_df))

print("\nFiles exist:",
      pedestrian_file.exists(),
      bicycle_file.exists())

Saved pedestrian data to: data/raw/pedestrian_raw.csv
Pedestrian records saved: 5000

Saved bicycle data to: data/raw/bicycle_raw.csv
Bicycle records saved: 352

Files exist: True True


### Observation

Both datasets have been successfully saved to local storage, confirming that the data collection pipeline is functioning correctly.  

The pedestrian dataset contains 5000 records, representing a sample of continuous sensor data, while the bicycle dataset contains 352 records, representing the complete dataset.  

These datasets are now ready for the next stage of preprocessing, including cleaning, transformation, and feature engineering.

## Final Summary

This section provides a summary of the data collection process, including the number of records retrieved and the file locations where the datasets have been stored.

In [12]:
# -----------------------------
# 12. Final summary
# -----------------------------
summary = {
    "Pedestrian records": len(pedestrian_df),
    "Bicycle records": len(bicycle_df),
    "Pedestrian file": str(pedestrian_file),
    "Bicycle file": str(bicycle_file),
}

for key, value in summary.items():
    print(f"{key}: {value}")

{'pedestrian_rows': 5000,
 'bicycle_rows': 352,
 'pedestrian_file': 'data/raw/pedestrian_raw.csv',
 'bicycle_file': 'data/raw/bicycle_raw.csv'}

### Conclusion

The data collection process has been successfully completed using the City of Melbourne Open Data API.  

Both pedestrian and bicycle datasets were retrieved, validated, and stored locally.  
The pipeline is reproducible and ready for the next stage, which involves data cleaning and preprocessing.